# Extraction de poses humaines depuis une vidéo
## MediaPipe Pose — Google Colab

Ce notebook montre comment extraire automatiquement les positions 3D des
articulations d'un humain dans une vidéo, sans GPU et sans installation complexe.

**Pipeline :**
```
Vidéo  →  MediaPipe Pose  →  33 landmarks 3D  →  Visualisation + Analyse
```

**Ce qu'on obtient :** pour chaque frame, les coordonnées (x, y, z) normalisées
de 33 points du corps (hanches, genoux, chevilles, épaules, coudes...).

In [ ]:
# Installation (à lancer une seule fois)
!pip install mediapipe opencv-python-headless matplotlib --quiet

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from google.colab import files
import os

# Raccourcis MediaPipe
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

# Noms des 33 landmarks MediaPipe (utile pour l'analyse)
LANDMARK_NAMES = [lm.name for lm in mp_pose.PoseLandmark]
print(f"MediaPipe version : {mp.__version__}")
print(f"Landmarks disponibles : {len(LANDMARK_NAMES)}")
print(LANDMARK_NAMES)

## 1. Charger une vidéo

Deux options : uploader votre propre vidéo, ou télécharger un exemple depuis YouTube.

In [ ]:
# --- Option A : uploader votre vidéo ---
# uploaded = files.upload()
# VIDEO_PATH = list(uploaded.keys())[0]

# --- Option B : télécharger un exemple (patinage) ---
!pip install yt-dlp --quiet
!yt-dlp -f "best[height<=480]" -o sample_video.mp4 \
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # ← remplacez par votre URL
VIDEO_PATH = "sample_video.mp4"

# Vérification
cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"Vidéo : {VIDEO_PATH}")
print(f"  {width}×{height}px  {fps:.1f}fps  {total} frames ({total/fps:.1f}s)")

## 2. Extraction des poses (toutes les frames)

In [ ]:
def extract_poses(video_path, max_frames=None):
    """
    Extrait les landmarks MediaPipe pour chaque frame de la vidéo.

    Retourne :
        landmarks  : np.array (T, 33, 4)  — x, y, z, visibility par frame
        frames_rgb : liste de T images RGB (pour visualisation)
    """
    landmarks_all = []
    frames_rgb    = []

    cap = cv2.VideoCapture(video_path)

    with mp_pose.Pose(
        model_complexity=1,       # 0=rapide, 1=équilibré, 2=précis
        smooth_landmarks=True,    # lissage temporel
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if max_frames and frame_idx >= max_frames:
                break

            # MediaPipe attend du RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results   = pose.process(frame_rgb)

            if results.pose_world_landmarks:
                # Coordonnées 3D monde (en mètres, centré sur les hanches)
                lm = results.pose_world_landmarks.landmark
                row = np.array([[l.x, l.y, l.z, l.visibility] for l in lm])  # (33, 4)
            else:
                row = np.zeros((33, 4))

            landmarks_all.append(row)
            frames_rgb.append(frame_rgb)
            frame_idx += 1

            if frame_idx % 50 == 0:
                print(f"  Frame {frame_idx}…", end="\r")

    cap.release()
    print(f"\nExtraction terminée : {frame_idx} frames")
    return np.array(landmarks_all), frames_rgb   # (T, 33, 4)


# Lance l'extraction (limité à 300 frames pour la démo)
landmarks, frames = extract_poses(VIDEO_PATH, max_frames=300)
print(f"Shape landmarks : {landmarks.shape}  (frames, articulations, xyzv)")

## 3. Visualisation — squelette sur une frame

In [ ]:
def draw_skeleton_on_frame(frame_rgb, video_path, frame_idx):
    """Redessine le squelette MediaPipe sur une frame."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with mp_pose.Pose(model_complexity=1, min_detection_confidence=0.5) as pose:
        results = pose.process(frame_rgb)

    annotated = frame_rgb.copy()
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            annotated,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
        )

    plt.figure(figsize=(10, 6))
    plt.imshow(annotated)
    plt.title(f"Frame {frame_idx} — squelette MediaPipe (33 landmarks)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


draw_skeleton_on_frame(frames[0], VIDEO_PATH, frame_idx=30)

## 4. Analyse — trajectoires des chevilles dans le temps

Pour le patinage, ce qui est intéressant c'est le **mouvement alterné des pieds**.

In [ ]:
# Indices MediaPipe utiles
LEFT_ANKLE  = mp_pose.PoseLandmark.LEFT_ANKLE.value   # 27
RIGHT_ANKLE = mp_pose.PoseLandmark.RIGHT_ANKLE.value  # 28
LEFT_HIP    = mp_pose.PoseLandmark.LEFT_HIP.value     # 23
RIGHT_HIP   = mp_pose.PoseLandmark.RIGHT_HIP.value    # 24

T   = len(landmarks)
fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS)
t   = np.arange(T) / fps   # axe temporel en secondes

# Positions 3D monde (en mètres, centré hanches)
l_ankle = landmarks[:, LEFT_ANKLE,  :3]   # (T, 3)
r_ankle = landmarks[:, RIGHT_ANKLE, :3]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
labels = ["X (gauche/droite)", "Y (haut/bas)", "Z (avant/arrière)"]
colors_l, colors_r = "royalblue", "tomato"

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(t, l_ankle[:, i], color=colors_l, lw=1.5, label="Cheville gauche")
    ax.plot(t, r_ankle[:, i], color=colors_r, lw=1.5, label="Cheville droite")
    ax.set_ylabel(label + " (m)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Temps (s)")
fig.suptitle("Trajectoires 3D des chevilles (repère monde MediaPipe)", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Visualisation 3D du squelette (une frame)

In [ ]:
# Connexions MediaPipe (paires d'indices)
CONNECTIONS = [(a.value, b.value) for a, b in mp_pose.POSE_CONNECTIONS]

def plot_skeleton_3d(landmarks_frame, title=""):
    """Visualise le squelette 3D pour une seule frame."""
    fig = plt.figure(figsize=(7, 9))
    ax  = fig.add_subplot(111, projection="3d")

    xyz = landmarks_frame[:, :3]   # (33, 3)
    vis = landmarks_frame[:, 3]    # visibilité

    # Points
    ax.scatter(xyz[:, 0], xyz[:, 2], -xyz[:, 1],
               c=vis, cmap="RdYlGn", s=30, vmin=0, vmax=1)

    # Os
    for a, b in CONNECTIONS:
        if vis[a] > 0.3 and vis[b] > 0.3:
            ax.plot([xyz[a,0], xyz[b,0]],
                    [xyz[a,2], xyz[b,2]],
                    [-xyz[a,1], -xyz[b,1]],
                    "gray", lw=1.5, alpha=0.6)

    ax.set_xlabel("X"); ax.set_ylabel("Z"); ax.set_zlabel("Y")
    ax.set_title(title or "Squelette 3D MediaPipe")
    plt.tight_layout()
    plt.show()


# Frame avec la meilleure détection (visibilité max)
best_frame = np.argmax(landmarks[:, :, 3].mean(axis=1))
print(f"Meilleure frame : {best_frame}")
plot_skeleton_3d(landmarks[best_frame], title=f"Frame {best_frame}")

## 6. Détecter le cycle de pas

La hauteur verticale des chevilles oscille en opposition de phase → on peut
détecter automatiquement la fréquence de foulée.

In [ ]:
from scipy.signal import find_peaks, butter, filtfilt

# Hauteur cheville gauche (axe Y MediaPipe = vers le bas → on inverse)
h_left  = -l_ankle[:, 1]   # positif = haut
h_right = -r_ankle[:, 1]

# Filtre passe-bas pour enlever le bruit haute fréquence
def smooth(signal, cutoff_hz=3.0, fs=fps):
    b, a = butter(2, cutoff_hz / (fs / 2), btype="low")
    return filtfilt(b, a, signal)

h_left_s  = smooth(h_left)
h_right_s = smooth(h_right)

# Détection des levées de pied (pics de hauteur)
peaks_l, _ = find_peaks(h_left_s,  height=np.percentile(h_left_s, 60),  distance=fps*0.3)
peaks_r, _ = find_peaks(h_right_s, height=np.percentile(h_right_s, 60), distance=fps*0.3)

plt.figure(figsize=(14, 4))
plt.plot(t, h_left_s,  color=colors_l, lw=1.5, label="Cheville gauche")
plt.plot(t, h_right_s, color=colors_r, lw=1.5, label="Cheville droite")
plt.scatter(t[peaks_l], h_left_s[peaks_l],   color=colors_l, s=80, zorder=5, label="Levée G")
plt.scatter(t[peaks_r], h_right_s[peaks_r],  color=colors_r, s=80, zorder=5, label="Levée D")
plt.xlabel("Temps (s)"); plt.ylabel("Hauteur cheville (m)")
plt.title("Cycle de pas — levées de pieds détectées")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Fréquence de foulée
if len(peaks_l) > 1:
    stride_times = np.diff(t[peaks_l])
    print(f"Foulée gauche  : {1/stride_times.mean():.2f} Hz  (période={stride_times.mean():.2f}s)")
if len(peaks_r) > 1:
    stride_times = np.diff(t[peaks_r])
    print(f"Foulée droite  : {1/stride_times.mean():.2f} Hz  (période={stride_times.mean():.2f}s)")

## Récapitulatif

| Ce qu'on a fait | Outil |
|----------------|-------|
| Extraction pose 3D | MediaPipe Pose |
| Visualisation squelette | OpenCV + Matplotlib |
| Trajectoires articulaires | NumPy |
| Détection cycle de pas | SciPy signal |

**Limites de MediaPipe** : coordonnées normalisées (pas en mètres absolus),
pas de paramètres SMPL, précision moindre que GVHMR sur les rotations articulaires.

**Pour aller plus loin** : GVHMR (SIGGRAPH Asia 2024) produit des paramètres SMPL
complets en coordonnées monde, utilisables pour l'animation physique.